In [21]:
import pandas as pd
import numpy as np
from tweetple import TweetPle
import sys
import pytz
from datetime import datetime, timedelta
import pandas as pd
from datetime import date
from tqdm import tqdm
from ast import literal_eval

sys.path.insert(0, '../../src/utils')
from funcs import *

sys.path.insert(0, '../../src/utils')
from general import *

### Followers Randomization BATCH 1

In [ ]:
followers_sa = pd.read_parquet('../../data/randomization/SA/influencers/followers_ties.parquet')
followers_sa['weak'] = np.where(followers_sa['strong']== 1, 0, followers_sa['weak'])

followers_grouped_n_sa = followers_sa[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_n_sa.rename({'influencer_id': 'n_following'}, axis=1, inplace=True)

followers_grouped_sa = followers_sa.groupby('follower_id').sum()
followers_grouped_sa = followers_grouped_sa.merge(followers_grouped_n_sa, on='follower_id', how='left')
followers_grouped_sa = followers_grouped_sa[['n_following','strong', 'weak']]
followers_grouped_sa['absent'] = followers_grouped_sa['n_following'] - followers_grouped_sa['strong'] - followers_grouped_sa['weak']

followers_ke = pd.read_parquet('../../data/randomization/KE/influencers/followers_ties.parquet')
followers_ke['weak'] = np.where(followers_ke['strong']== 1, 0, followers_ke['weak'])

followers_grouped_n_ke = followers_ke[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_n_ke.rename({'influencer_id': 'n_following'}, axis=1, inplace=True)

followers_grouped_ke = followers_ke.groupby('follower_id').sum()
followers_grouped_ke = followers_grouped_ke.merge(followers_grouped_n_ke, on='follower_id', how='left')
followers_grouped_ke = followers_grouped_ke[['n_following','strong', 'weak']]
followers_grouped_ke['absent'] = followers_grouped_ke['n_following'] - followers_grouped_ke['strong'] - followers_grouped_ke['weak']

followers_grouped_sa.reset_index(inplace=True)
followers_grouped_ke.reset_index(inplace=True)
followers_grouped_ke.rename(columns = {'followers':'n_followers',
                            'strong':'n_strong',
                            'weak':'n_weak',
                            'absent':'n_absent'}, inplace = True)
followers_grouped_sa.rename(columns = {'followers':'n_followers',
                            'strong':'n_strong',
                            'weak':'n_weak',
                            'absent':'n_absent'}, inplace = True)

followers_grouped_ke['follower_id'] = followers_grouped_ke['follower_id'].astype(str)
followers_grouped_sa['follower_id'] = followers_grouped_sa['follower_id'].astype(str)

b_path = f'../../data/randomization/KE/followers/integrate/'
file = 'followers.parquet.gzip'
info_followers_ke = pd.read_parquet(f'{b_path}{file}')

b_path = f'../../data/randomization/SA/followers/integrate/'
file = 'followers.parquet.gzip'
info_followers_sa = pd.read_parquet(f'{b_path}{file}')

info_followers_sa = info_followers_sa.join(
      pd.json_normalize(info_followers_sa['public_metrics'])
 ).drop('public_metrics',axis=1)

info_followers_ke = info_followers_ke.join(
      pd.json_normalize(info_followers_ke['public_metrics'])
 ).drop('public_metrics',axis=1)

info_followers_sa.drop_duplicates('id', keep = 'last', inplace=True)
info_followers_ke.drop_duplicates('id', keep = 'last', inplace=True)

info_followers_ke.reset_index(drop=True, inplace=True)
info_followers_sa.reset_index(drop=True, inplace=True)

ids_ke = list(info_followers_ke['id'].astype(str))
ids_sa = list(info_followers_sa['id'].astype(str))

info_followers_ke.rename(columns={'id':'follower_id',
                                  'public_metrics.followers_count':'followers_count',
                                  'public_metrics.following_count': 'following_count',
                                  'public_metrics.tweet_count':'tweet_count',
                                  'public_metrics.listed_count':'listed_count',}, inplace=True)
info_followers_sa.rename(columns={'id':'follower_id',
                                  'public_metrics.followers_count':'followers_count',
                                  'public_metrics.following_count': 'following_count',
                                  'public_metrics.tweet_count':'tweet_count',
                                  'public_metrics.listed_count':'listed_count',}, inplace=True)

info_followers_ke = info_followers_ke.merge(followers_grouped_ke,on='follower_id',how='left')
info_followers_sa = info_followers_sa.merge(followers_grouped_sa,on='follower_id',how='left')

info_followers_ke.to_parquet('../../data/randomization/KE/02-variables/f_variables_followers.parquet')
info_followers_sa.to_parquet('../../data/randomization/SA/02-variables/f_variables_followers.parquet')

treatment_ke = pd.read_excel('../../data/randomization/KE/03-assignment/output/RandomizedTwitterSampleKE.xlsx')
treatment_sa = pd.read_excel('../../data/randomization/SA/03-assignment/output/RandomizedTwitterSampleSA.xlsx')

treatment_ke['author_id'] = treatment_ke['author_id'].astype(str)
treatment_sa['author_id'] = treatment_sa['author_id'].astype(str)

control_ke = list(treatment_ke[treatment_ke['treatment']==0].author_id)
control_sa = list(treatment_sa[treatment_sa['treatment']==0].author_id)

treat_ke = list(treatment_ke[treatment_ke['treatment']==1].author_id)
treat_sa = list(treatment_sa[treatment_sa['treatment']==1].author_id)

info_followers_ke = pd.read_parquet('../../data/randomization/KE/02-variables/f_variables_followers.parquet')
info_followers_sa = pd.read_parquet('../../data/randomization/SA/02-variables/f_variables_followers.parquet')

followers_sa = pd.read_parquet('../../data/randomization/SA/influencers/followers_ties.parquet')
followers_ke =  pd.read_parquet('../../data/randomization/KE/influencers/followers_ties.parquet')

followers_ke['weak'] = np.where(followers_ke['strong']== 1, 0, followers_ke['weak'])
followers_sa['weak'] = np.where(followers_sa['strong']== 1, 0, followers_sa['weak'])

followers_sa = followers_sa[followers_sa['influencer_id'].isin(treat_sa)]
followers_ke = followers_ke[followers_ke['influencer_id'].isin(treat_ke)]

followers_grouped_t_sa = followers_sa[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_t_sa.rename({'influencer_id': 'n_following_treated'}, axis=1, inplace=True)

followers_grouped_sa_t = followers_sa.groupby('follower_id').sum()
followers_grouped_sa_t = followers_grouped_sa_t.merge(followers_grouped_t_sa, on='follower_id', how='left')
followers_grouped_sa_t = followers_grouped_sa_t[['n_following_treated','strong', 'weak']]
followers_grouped_sa_t['absent'] = followers_grouped_sa_t['n_following_treated'] - followers_grouped_sa_t['strong'] - followers_grouped_sa_t['weak']

followers_grouped_t_ke = followers_ke[followers_ke['influencer_id'].isin(treat_ke)]
followers_grouped_t_ke = followers_grouped_t_ke[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_t_ke.rename({'influencer_id': 'n_following_treated'}, axis=1, inplace=True)

followers_grouped_ke_t = followers_ke[followers_ke['influencer_id'].isin(treat_ke)]
followers_grouped_ke_t = followers_grouped_ke_t.groupby('follower_id').sum()
followers_grouped_ke_t = followers_grouped_ke_t.merge(followers_grouped_t_ke, on='follower_id', how='left')
followers_grouped_ke_t = followers_grouped_ke_t[['n_following_treated','strong', 'weak']]
followers_grouped_ke_t['absent'] = followers_grouped_ke_t['n_following_treated'] - followers_grouped_ke_t['strong'] - followers_grouped_ke_t['weak']

followers_grouped_sa_t.reset_index(inplace=True)
followers_grouped_ke_t.reset_index(inplace=True)
followers_grouped_ke_t.rename(columns = {'followers':'n_followers_treated',
                            'strong':'n_strong_treated',
                            'weak':'n_weak_treated',
                            'absent':'n_absent_treated'}, inplace = True)
followers_grouped_sa_t.rename(columns = {'followers':'n_followers_treated',
                            'strong':'n_strong_treated',
                            'weak':'n_weak_treated',
                            'absent':'n_absent_treated'}, inplace = True)

followers_grouped_ke_t['follower_id'] = followers_grouped_ke_t['follower_id'].astype(str)
followers_grouped_sa_t['follower_id'] = followers_grouped_sa_t['follower_id'].astype(str)

info_followers_ke = info_followers_ke.merge(followers_grouped_ke_t,on='follower_id',how='left')
info_followers_sa = info_followers_sa.merge(followers_grouped_sa_t,on='follower_id',how='left')

followers_sa = pd.read_parquet('../../data/randomization/SA/influencers/followers_ties.parquet')
followers_ke =  pd.read_parquet('../../data/randomization/KE/influencers/followers_ties.parquet')

followers_ke['weak'] = np.where(followers_ke['strong']== 1, 0, followers_ke['weak'])
followers_sa['weak'] = np.where(followers_sa['strong']== 1, 0, followers_sa['weak'])

followers_sa = followers_sa[followers_sa['influencer_id'].isin(control_sa)]
followers_ke = followers_ke[followers_ke['influencer_id'].isin(control_ke)]

followers_grouped_c_sa = followers_sa[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_c_sa.rename({'influencer_id': 'n_following_control'}, axis=1, inplace=True)


followers_grouped_sa_c = followers_sa.groupby('follower_id').sum()
followers_grouped_sa_c = followers_grouped_sa_c.merge(followers_grouped_c_sa, on='follower_id', how='left')
followers_grouped_sa_c = followers_grouped_sa_c[['n_following_control','strong', 'weak']]
followers_grouped_sa_c['absent'] = followers_grouped_sa_c['n_following_control'] - followers_grouped_sa_c['strong'] - followers_grouped_sa_c['weak']

followers_grouped_c_ke = followers_ke[followers_ke['influencer_id'].isin(control_ke)]
followers_grouped_c_ke = followers_grouped_c_ke[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_c_ke.rename({'influencer_id': 'n_following_control'}, axis=1, inplace=True)


followers_grouped_ke_c = followers_ke[followers_ke['influencer_id'].isin(control_ke)]
followers_grouped_ke_c = followers_grouped_ke_c.groupby('follower_id').sum()
followers_grouped_ke_c = followers_grouped_ke_c.merge(followers_grouped_c_ke, on='follower_id', how='left')
followers_grouped_ke_c = followers_grouped_ke_c[['n_following_control','strong', 'weak']]
followers_grouped_ke_c['absent'] = followers_grouped_ke_c['n_following_control'] - followers_grouped_ke_c['strong'] - followers_grouped_ke_c['weak']


followers_grouped_sa_c.reset_index(inplace=True)
followers_grouped_ke_c.reset_index(inplace=True)
followers_grouped_ke_c.rename(columns = {'followers':'n_followers_control',
                            'strong':'n_strong_control',
                            'weak':'n_weak_control',
                            'absent':'n_absent_control'}, inplace = True)
followers_grouped_sa_c.rename(columns = {'followers':'n_followers_control',
                            'strong':'n_strong_control',
                            'weak':'n_weak_control',
                            'absent':'n_absent_control'}, inplace = True)

followers_grouped_ke_c['follower_id'] = followers_grouped_ke_c['follower_id'].astype(str)
followers_grouped_sa_c['follower_id'] = followers_grouped_sa_c['follower_id'].astype(str)

info_followers_ke = info_followers_ke.merge(followers_grouped_ke_c,on='follower_id',how='left')
info_followers_sa = info_followers_sa.merge(followers_grouped_sa_c,on='follower_id',how='left')

info_followers_ke.drop('author_id_following', axis=1, inplace=True)
info_followers_sa.drop('author_id_following', axis=1, inplace=True)

info_followers_ke["n_following_treated"] = info_followers_ke["n_following_treated"].fillna(0)
info_followers_ke["n_strong_treated"] = info_followers_ke["n_strong_treated"].fillna(0)
info_followers_ke["n_weak_treated"] = info_followers_ke["n_weak_treated"].fillna(0)
info_followers_ke["n_absent_treated"] = info_followers_ke["n_absent_treated"].fillna(0)
info_followers_ke["n_following_control"] = info_followers_ke["n_following_control"].fillna(0)
info_followers_ke["n_strong_control"] = info_followers_ke["n_strong_control"].fillna(0)
info_followers_ke["n_weak_control"] = info_followers_ke["n_weak_control"].fillna(0)
info_followers_ke["n_absent_control"] = info_followers_ke["n_absent_control"].fillna(0)

info_followers_sa["n_following_treated"] = info_followers_sa["n_following_treated"].fillna(0)
info_followers_sa["n_strong_treated"] = info_followers_sa["n_strong_treated"].fillna(0)
info_followers_sa["n_weak_treated"] = info_followers_sa["n_weak_treated"].fillna(0)
info_followers_sa["n_absent_treated"] = info_followers_sa["n_absent_treated"].fillna(0)
info_followers_sa["n_following_control"] = info_followers_sa["n_following_control"].fillna(0)
info_followers_sa["n_strong_control"] = info_followers_sa["n_strong_control"].fillna(0)
info_followers_sa["n_weak_control"] = info_followers_sa["n_weak_control"].fillna(0)
info_followers_sa["n_absent_control"] = info_followers_sa["n_absent_control"].fillna(0)

info_followers_ke['date_today'] = '2023-03-10'
info_followers_sa['date_today'] = '2023-03-10'

info_followers_ke['created_at'] = pd.to_datetime(info_followers_ke['created_at'], utc=True)
info_followers_ke['date_today'] = pd.to_datetime(info_followers_ke['date_today'], utc=True)
info_followers_ke['days_old_account'] = (info_followers_ke['date_today'] - info_followers_ke['created_at']).dt.days

info_followers_sa['created_at'] = pd.to_datetime(info_followers_sa['created_at'], utc=True)
info_followers_sa['date_today'] = pd.to_datetime(info_followers_sa['date_today'], utc=True)
info_followers_sa['days_old_account'] = (info_followers_sa['date_today'] - info_followers_sa['created_at']).dt.days

C:\Users\Dell\AppData\Local\Temp\ipykernel_36040\2712450153.py:7: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  followers_grouped_sa = followers_sa.groupby('follower_id').sum()
C:\Users\Dell\AppData\Local\Temp\ipykernel_36040\2712450153.py:18: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  followers_grouped_ke = followers_ke.groupby('follower_id').sum()
C:\Users\Dell\AppData\Local\Temp\ipykernel_36040\2712450153.py:106: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only colum

In [23]:
# Replication check for sanity: 
infK = pd.read_parquet('../../data/randomization/KE/02-variables/before_rep_variables_followers.parquet')
infS = pd.read_parquet('../../data/randomization/SA/02-variables/before_rep_variables_followers.parquet')

# Check follower_id sets match
ke_ids_match = set(infK['follower_id'].astype(str)) == set(info_followers_ke['follower_id'].astype(str))
sa_ids_match = set(infS['follower_id'].astype(str)) == set(info_followers_sa['follower_id'].astype(str))
print("KE follower_id sets match:", ke_ids_match)
print("SA follower_id sets match:", sa_ids_match)

# Compare n_strong and n_strong_treated on shared follower_ids
cols = ['follower_id', 'n_strong', 'n_strong_treated']

ke_old = infK[cols].copy()
ke_old['follower_id'] = ke_old['follower_id'].astype(str)
ke_new = info_followers_ke[cols].copy()
ke_new['follower_id'] = ke_new['follower_id'].astype(str)

sa_old = infS[cols].copy()
sa_old['follower_id'] = sa_old['follower_id'].astype(str)
sa_new = info_followers_sa[cols].copy()
sa_new['follower_id'] = sa_new['follower_id'].astype(str)

ke_merged = ke_old.merge(ke_new, on='follower_id', suffixes=('_old', '_new'))
sa_merged = sa_old.merge(sa_new, on='follower_id', suffixes=('_old', '_new'))

print("\nKE n_strong match:", (ke_merged['n_strong_old'] == ke_merged['n_strong_new']).all())
print("KE n_strong_treated match:", (ke_merged['n_strong_treated_old'] == ke_merged['n_strong_treated_new']).all())
print("\nSA n_strong match:", (sa_merged['n_strong_old'] == sa_merged['n_strong_new']).all())
print("SA n_strong_treated match:", (sa_merged['n_strong_treated_old'] == sa_merged['n_strong_treated_new']).all())


KE follower_id sets match: True
SA follower_id sets match: True

KE n_strong match: True
KE n_strong_treated match: True

SA n_strong match: True
SA n_strong_treated match: True


In [24]:
info_followers_ke.to_parquet('../../data/randomization/KE/02-variables/variables_followers.parquet')
info_followers_sa.to_parquet('../../data/randomization/SA/02-variables/variables_followers.parquet')

## Followers Ranomization Batch 2

In [ ]:
followers_sa = pd.read_parquet('../../data/randomization/SA/influencers/followers_ties_batch2.parquet')
followers_sa['weak'] = np.where(followers_sa['strong']== 1, 0, followers_sa['weak'])

followers_grouped_n_sa = followers_sa[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_n_sa.rename({'influencer_id': 'n_following'}, axis=1, inplace=True)

followers_grouped_sa = followers_sa.groupby('follower_id').sum()
followers_grouped_sa = followers_grouped_sa.merge(followers_grouped_n_sa, on='follower_id', how='left')
followers_grouped_sa = followers_grouped_sa[['n_following','strong', 'weak']]
followers_grouped_sa['absent'] = followers_grouped_sa['n_following'] - followers_grouped_sa['strong'] - followers_grouped_sa['weak']

followers_ke = pd.read_parquet('../../data/randomization/KE/influencers/followers_ties_batch2.parquet')
followers_ke['weak'] = np.where(followers_ke['strong']== 1, 0, followers_ke['weak'])

followers_grouped_n_ke = followers_ke[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_n_ke.rename({'influencer_id': 'n_following'}, axis=1, inplace=True)

followers_grouped_ke = followers_ke.groupby('follower_id').sum()
followers_grouped_ke = followers_grouped_ke.merge(followers_grouped_n_ke, on='follower_id', how='left')
followers_grouped_ke = followers_grouped_ke[['n_following','strong', 'weak']]
followers_grouped_ke['absent'] = followers_grouped_ke['n_following'] - followers_grouped_ke['strong'] - followers_grouped_ke['weak']

followers_grouped_sa.reset_index(inplace=True)
followers_grouped_ke.reset_index(inplace=True)
followers_grouped_ke.rename(columns = {'followers':'n_followers',
                            'strong':'n_strong',
                            'weak':'n_weak',
                            'absent':'n_absent'}, inplace = True)
followers_grouped_sa.rename(columns = {'followers':'n_followers',
                            'strong':'n_strong',
                            'weak':'n_weak',
                            'absent':'n_absent'}, inplace = True)

followers_grouped_ke['follower_id'] = followers_grouped_ke['follower_id'].astype(str)
followers_grouped_sa['follower_id'] = followers_grouped_sa['follower_id'].astype(str)

b_path = f'../../data/randomization/KE/followers/integrate/'
file = 'followers_batch2.parquet.gzip'
info_followers_ke = pd.read_parquet(f'{b_path}{file}')

b_path = f'../../data/randomization/SA/followers/integrate/'
file = 'followers_batch2.parquet.gzip'
info_followers_sa = pd.read_parquet(f'{b_path}{file}')

info_followers_sa = info_followers_sa.join(
      pd.json_normalize(info_followers_sa['public_metrics'])
 ).drop('public_metrics',axis=1)

info_followers_ke = info_followers_ke.join(
      pd.json_normalize(info_followers_ke['public_metrics'])
 ).drop('public_metrics',axis=1)

info_followers_sa.drop_duplicates('id', keep = 'last', inplace=True)
info_followers_ke.drop_duplicates('id', keep = 'last', inplace=True)

info_followers_ke.reset_index(drop=True, inplace=True)
info_followers_sa.reset_index(drop=True, inplace=True)

ids_ke = list(info_followers_ke['id'].astype(str))
ids_sa = list(info_followers_sa['id'].astype(str))

info_followers_ke.rename(columns={'id':'follower_id',
                                  'public_metrics.followers_count':'followers_count',
                                  'public_metrics.following_count': 'following_count',
                                  'public_metrics.tweet_count':'tweet_count',
                                  'public_metrics.listed_count':'listed_count',}, inplace=True)
info_followers_sa.rename(columns={'id':'follower_id',
                                  'public_metrics.followers_count':'followers_count',
                                  'public_metrics.following_count': 'following_count',
                                  'public_metrics.tweet_count':'tweet_count',
                                  'public_metrics.listed_count':'listed_count',}, inplace=True)

info_followers_ke = info_followers_ke.merge(followers_grouped_ke,on='follower_id',how='left')
info_followers_sa = info_followers_sa.merge(followers_grouped_sa,on='follower_id',how='left')

info_followers_ke.to_parquet('../../data/randomization/KE/02-variables/f_variables_followers_batch2.parquet')
info_followers_sa.to_parquet('../../data/randomization/SA/02-variables/f_variables_followers_batch2.parquet')

treatment_ke = pd.read_excel('../../data/randomization/KE/03-assignment/output/RandomizedTwitterSampleKE_batch2.xlsx')
treatment_sa = pd.read_excel('../../data/randomization/SA/03-assignment/output/RandomizedTwitterSampleSA_batch2.xlsx')

treatment_ke['author_id'] = treatment_ke['author_id'].astype(str)
treatment_sa['author_id'] = treatment_sa['author_id'].astype(str)

control_ke = list(treatment_ke[treatment_ke['treatment']==0].author_id)
control_sa = list(treatment_sa[treatment_sa['treatment']==0].author_id)

treat_ke = list(treatment_ke[treatment_ke['treatment']==1].author_id)
treat_sa = list(treatment_sa[treatment_sa['treatment']==1].author_id)

info_followers_ke = pd.read_parquet('../../data/randomization/KE/02-variables/f_variables_followers_batch2.parquet')
info_followers_sa = pd.read_parquet('../../data/randomization/SA/02-variables/f_variables_followers_batch2.parquet')

followers_sa = pd.read_parquet('../../data/randomization/SA/influencers/followers_ties_batch2.parquet')
followers_ke =  pd.read_parquet('../../data/randomization/KE/influencers/followers_ties_batch2.parquet')

followers_ke['weak'] = np.where(followers_ke['strong']== 1, 0, followers_ke['weak'])
followers_sa['weak'] = np.where(followers_sa['strong']== 1, 0, followers_sa['weak'])

followers_sa = followers_sa[followers_sa['influencer_id'].isin(treat_sa)]
followers_ke = followers_ke[followers_ke['influencer_id'].isin(treat_ke)]

followers_grouped_t_sa = followers_sa[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_t_sa.rename({'influencer_id': 'n_following_treated'}, axis=1, inplace=True)

followers_grouped_sa_t = followers_sa.groupby('follower_id').sum()
followers_grouped_sa_t = followers_grouped_sa_t.merge(followers_grouped_t_sa, on='follower_id', how='left')
followers_grouped_sa_t = followers_grouped_sa_t[['n_following_treated','strong', 'weak']]
followers_grouped_sa_t['absent'] = followers_grouped_sa_t['n_following_treated'] - followers_grouped_sa_t['strong'] - followers_grouped_sa_t['weak']

followers_grouped_t_ke = followers_ke[followers_ke['influencer_id'].isin(treat_ke)]
followers_grouped_t_ke = followers_grouped_t_ke[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_t_ke.rename({'influencer_id': 'n_following_treated'}, axis=1, inplace=True)

followers_grouped_ke_t = followers_ke[followers_ke['influencer_id'].isin(treat_ke)]
followers_grouped_ke_t = followers_grouped_ke_t.groupby('follower_id').sum()
followers_grouped_ke_t = followers_grouped_ke_t.merge(followers_grouped_t_ke, on='follower_id', how='left')
followers_grouped_ke_t = followers_grouped_ke_t[['n_following_treated','strong', 'weak']]
followers_grouped_ke_t['absent'] = followers_grouped_ke_t['n_following_treated'] - followers_grouped_ke_t['strong'] - followers_grouped_ke_t['weak']

followers_grouped_sa_t.reset_index(inplace=True)
followers_grouped_ke_t.reset_index(inplace=True)
followers_grouped_ke_t.rename(columns = {'followers':'n_followers_treated',
                            'strong':'n_strong_treated',
                            'weak':'n_weak_treated',
                            'absent':'n_absent_treated'}, inplace = True)
followers_grouped_sa_t.rename(columns = {'followers':'n_followers_treated',
                            'strong':'n_strong_treated',
                            'weak':'n_weak_treated',
                            'absent':'n_absent_treated'}, inplace = True)

followers_grouped_ke_t['follower_id'] = followers_grouped_ke_t['follower_id'].astype(str)
followers_grouped_sa_t['follower_id'] = followers_grouped_sa_t['follower_id'].astype(str)

info_followers_ke = info_followers_ke.merge(followers_grouped_ke_t,on='follower_id',how='left')
info_followers_sa = info_followers_sa.merge(followers_grouped_sa_t,on='follower_id',how='left')

followers_sa = pd.read_parquet('../../data/randomization/SA/influencers/followers_ties_batch2.parquet')
followers_ke =  pd.read_parquet('../../data/randomization/KE/influencers/followers_ties_batch2.parquet')

followers_ke['weak'] = np.where(followers_ke['strong']== 1, 0, followers_ke['weak'])
followers_sa['weak'] = np.where(followers_sa['strong']== 1, 0, followers_sa['weak'])

followers_sa = followers_sa[followers_sa['influencer_id'].isin(control_sa)]
followers_ke = followers_ke[followers_ke['influencer_id'].isin(control_ke)]

followers_grouped_c_sa = followers_sa[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_c_sa.rename({'influencer_id': 'n_following_control'}, axis=1, inplace=True)


followers_grouped_sa_c = followers_sa.groupby('follower_id').sum()
followers_grouped_sa_c = followers_grouped_sa_c.merge(followers_grouped_c_sa, on='follower_id', how='left')
followers_grouped_sa_c = followers_grouped_sa_c[['n_following_control','strong', 'weak']]
followers_grouped_sa_c['absent'] = followers_grouped_sa_c['n_following_control'] - followers_grouped_sa_c['strong'] - followers_grouped_sa_c['weak']

followers_grouped_c_ke = followers_ke[followers_ke['influencer_id'].isin(control_ke)]
followers_grouped_c_ke = followers_grouped_c_ke[['influencer_id', 'follower_id']].groupby('follower_id').count()
followers_grouped_c_ke.rename({'influencer_id': 'n_following_control'}, axis=1, inplace=True)


followers_grouped_ke_c = followers_ke[followers_ke['influencer_id'].isin(control_ke)]
followers_grouped_ke_c = followers_grouped_ke_c.groupby('follower_id').sum()
followers_grouped_ke_c = followers_grouped_ke_c.merge(followers_grouped_c_ke, on='follower_id', how='left')
followers_grouped_ke_c = followers_grouped_ke_c[['n_following_control','strong', 'weak']]
followers_grouped_ke_c['absent'] = followers_grouped_ke_c['n_following_control'] - followers_grouped_ke_c['strong'] - followers_grouped_ke_c['weak']


followers_grouped_sa_c.reset_index(inplace=True)
followers_grouped_ke_c.reset_index(inplace=True)
followers_grouped_ke_c.rename(columns = {'followers':'n_followers_control',
                            'strong':'n_strong_control',
                            'weak':'n_weak_control',
                            'absent':'n_absent_control'}, inplace = True)
followers_grouped_sa_c.rename(columns = {'followers':'n_followers_control',
                            'strong':'n_strong_control',
                            'weak':'n_weak_control',
                            'absent':'n_absent_control'}, inplace = True)

followers_grouped_ke_c['follower_id'] = followers_grouped_ke_c['follower_id'].astype(str)
followers_grouped_sa_c['follower_id'] = followers_grouped_sa_c['follower_id'].astype(str)

info_followers_ke = info_followers_ke.merge(followers_grouped_ke_c,on='follower_id',how='left')
info_followers_sa = info_followers_sa.merge(followers_grouped_sa_c,on='follower_id',how='left')

info_followers_ke.drop('author_id_following', axis=1, inplace=True)
info_followers_sa.drop('author_id_following', axis=1, inplace=True)

info_followers_ke["n_following_treated"] = info_followers_ke["n_following_treated"].fillna(0)
info_followers_ke["n_strong_treated"] = info_followers_ke["n_strong_treated"].fillna(0)
info_followers_ke["n_weak_treated"] = info_followers_ke["n_weak_treated"].fillna(0)
info_followers_ke["n_absent_treated"] = info_followers_ke["n_absent_treated"].fillna(0)
info_followers_ke["n_following_control"] = info_followers_ke["n_following_control"].fillna(0)
info_followers_ke["n_strong_control"] = info_followers_ke["n_strong_control"].fillna(0)
info_followers_ke["n_weak_control"] = info_followers_ke["n_weak_control"].fillna(0)
info_followers_ke["n_absent_control"] = info_followers_ke["n_absent_control"].fillna(0)

info_followers_sa["n_following_treated"] = info_followers_sa["n_following_treated"].fillna(0)
info_followers_sa["n_strong_treated"] = info_followers_sa["n_strong_treated"].fillna(0)
info_followers_sa["n_weak_treated"] = info_followers_sa["n_weak_treated"].fillna(0)
info_followers_sa["n_absent_treated"] = info_followers_sa["n_absent_treated"].fillna(0)
info_followers_sa["n_following_control"] = info_followers_sa["n_following_control"].fillna(0)
info_followers_sa["n_strong_control"] = info_followers_sa["n_strong_control"].fillna(0)
info_followers_sa["n_weak_control"] = info_followers_sa["n_weak_control"].fillna(0)
info_followers_sa["n_absent_control"] = info_followers_sa["n_absent_control"].fillna(0)

info_followers_ke['date_today'] = '2023-03-10'
info_followers_sa['date_today'] = '2023-03-10'

info_followers_ke['created_at'] = pd.to_datetime(info_followers_ke['created_at'], utc=True)
info_followers_ke['date_today'] = pd.to_datetime(info_followers_ke['date_today'], utc=True)
info_followers_ke['days_old_account'] = (info_followers_ke['date_today'] - info_followers_ke['created_at']).dt.days

info_followers_sa['created_at'] = pd.to_datetime(info_followers_sa['created_at'], utc=True)
info_followers_sa['date_today'] = pd.to_datetime(info_followers_sa['date_today'], utc=True)
info_followers_sa['days_old_account'] = (info_followers_sa['date_today'] - info_followers_sa['created_at']).dt.days